# Ray XGBoost Training on EKS

This notebook demonstrates how to train XGBoost models using Ray on EKS for fraud detection.


In [ ]:
import ray
import pandas as pd
import numpy as np
import xgboost as xgb
import boto3
import os
import joblib
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import ray.train.xgboost as ray_xgboost
from ray.train import ScalingConfig
from ray.air import session

# Configuration
RAY_ADDRESS = os.environ.get('RAY_ADDRESS', 'ray://ray-cluster-head-svc.ray-system.svc.cluster.local:10001')
S3_BUCKET = os.environ['S3_BUCKET']
AWS_REGION = os.environ['AWS_DEFAULT_REGION']

print(f"Ray Address: {RAY_ADDRESS}")
print(f"S3 Bucket: {S3_BUCKET}")
print(f"AWS Region: {AWS_REGION}")

## Connect to Ray Cluster


In [ ]:
# Connect to Ray cluster
try:
    ray.shutdown()  # Shutdown any existing connection
except:
    pass

ray.init(address=RAY_ADDRESS)

print("Ray cluster info:")
print(ray.cluster_resources())
print(f"\nRay dashboard: {ray.get_dashboard_url()}")

## Load and Prepare Data


In [ ]:
# Initialize S3 client
s3_client = boto3.client('s3', region_name=AWS_REGION)

@ray.remote
def load_data_from_s3(bucket, key):
    """Load data from S3 using Ray remote function"""
    import pandas as pd
    import boto3
    
    s3_client = boto3.client('s3')
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    df = pd.read_parquet(obj['Body'])
    return df

# Load processed fraud detection data
data_key = "data/output/fraud_features.parquet"

try:
    # Check if processed data exists
    s3_client.head_object(Bucket=S3_BUCKET, Key=data_key)
    print(f"Loading data from s3://{S3_BUCKET}/{data_key}")
    
    # Load data using Ray
    df_future = load_data_from_s3.remote(S3_BUCKET, data_key)
    df = ray.get(df_future)
    
    print(f"Data loaded: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nTarget distribution:")
    print(df['TX_FRAUD_1'].value_counts())
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Using synthetic data for demonstration...")
    
    # Create synthetic fraud detection data
    np.random.seed(42)
    n_samples = 10000
    
    df = pd.DataFrame({
        'TX_AMOUNT': np.random.lognormal(3, 1, n_samples),
        'customer_id_nb_txns_1_window': np.random.poisson(5, n_samples),
        'customer_id_avg_amt_1_window': np.random.lognormal(3, 0.5, n_samples),
        'terminal_id_nb_txns_1_window': np.random.poisson(20, n_samples),
        'terminal_id_avg_amt_1_window': np.random.lognormal(3, 0.3, n_samples),
        'customer_id_nb_txns_7_window': np.random.poisson(35, n_samples),
        'customer_id_avg_amt_7_window': np.random.lognormal(3, 0.4, n_samples),
        'terminal_id_nb_txns_7_window': np.random.poisson(140, n_samples),
        'terminal_id_avg_amt_7_window': np.random.lognormal(3, 0.2, n_samples),
    })
    
    # Create fraud labels (5% fraud rate)
    fraud_prob = 0.05 + 0.1 * (df['TX_AMOUNT'] > df['TX_AMOUNT'].quantile(0.95)).astype(int)
    df['TX_FRAUD_1'] = np.random.binomial(1, fraud_prob)
    
    print(f"Synthetic data created: {df.shape}")
    print(f"Fraud rate: {df['TX_FRAUD_1'].mean():.3f}")

## Prepare Training Data


In [ ]:
# Prepare features and target
feature_columns = [col for col in df.columns if col != 'TX_FRAUD_1']
X = df[feature_columns]
y = df['TX_FRAUD_1']

print(f"Features: {feature_columns}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training fraud rate: {y_train.mean():.3f}")
print(f"Test fraud rate: {y_test.mean():.3f}")

## Define Ray Training Function


In [ ]:
def train_xgboost_model(config):
    """Ray training function for XGBoost"""
    import xgboost as xgb
    import pandas as pd
    from ray.air import session
    from sklearn.metrics import roc_auc_score, classification_report
    
    # Get training data from session
    train_set = session.get_dataset_shard("train")
    valid_set = session.get_dataset_shard("valid")
    
    # Convert to pandas DataFrames
    train_df = train_set.to_pandas()
    valid_df = valid_set.to_pandas()
    
    # Separate features and target
    feature_cols = [col for col in train_df.columns if col != 'TX_FRAUD_1']
    
    X_train = train_df[feature_cols]
    y_train = train_df['TX_FRAUD_1']
    X_valid = valid_df[feature_cols]
    y_valid = valid_df['TX_FRAUD_1']
    
    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    
    # XGBoost parameters
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'gpu_hist' if config.get('use_gpu', False) else 'hist',
        'max_depth': config.get('max_depth', 6),
        'learning_rate': config.get('learning_rate', 0.1),
        'subsample': config.get('subsample', 0.8),
        'colsample_bytree': config.get('colsample_bytree', 0.8),
        'scale_pos_weight': config.get('scale_pos_weight', 1),
        'random_state': 42
    }
    
    # Training callback for Ray
    def ray_callback(env):
        # Report metrics to Ray
        if env.evaluation_result_list:
            for item in env.evaluation_result_list:
                if len(item) == 3:  # (dataset_name, metric_name, value)
                    dataset_name, metric_name, value = item
                    session.report({f"{dataset_name}_{metric_name}": value})
    
    # Train model
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=config.get('num_boost_round', 100),
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        callbacks=[ray_callback],
        verbose_eval=False
    )
    
    # Make predictions
    y_pred_proba = model.predict(dvalid)
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    # Calculate metrics
    auc_score = roc_auc_score(y_valid, y_pred_proba)
    
    # Report final metrics
    session.report({
        'final_auc': auc_score,
        'final_accuracy': (y_pred == y_valid).mean()
    })
    
    return model

print("Training function defined")

## Create Ray Datasets


In [ ]:
# Create Ray datasets
train_df = pd.concat([X_train, y_train], axis=1)
valid_df = pd.concat([X_test, y_test], axis=1)

train_dataset = ray.data.from_pandas(train_df)
valid_dataset = ray.data.from_pandas(valid_df)

print(f"Train dataset: {train_dataset.count()} rows")
print(f"Valid dataset: {valid_dataset.count()} rows")

## Train Model with Ray


In [ ]:
from ray.train import Trainer
from ray.air.config import ScalingConfig

# Training configuration
training_config = {
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),  # Handle class imbalance
    'num_boost_round': 100,
    'use_gpu': True  # Enable GPU training if available
}

# Scaling configuration - use multiple workers for distributed training
scaling_config = ScalingConfig(
    num_workers=2,  # Number of training workers
    use_gpu=True,   # Use GPU if available
    resources_per_worker={"CPU": 2, "GPU": 1}
)

print(f"Training configuration: {training_config}")
print(f"Scaling configuration: {scaling_config}")

# Create trainer
trainer = ray_xgboost.XGBoostTrainer(
    scaling_config=scaling_config,
    label_column="TX_FRAUD_1",
    params={
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'gpu_hist',
        **{k: v for k, v in training_config.items() if k not in ['num_boost_round', 'use_gpu']}
    },
    num_boost_round=training_config['num_boost_round'],
    datasets={"train": train_dataset, "valid": valid_dataset}
)

print("Starting distributed training...")
start_time = datetime.now()

# Train the model
result = trainer.fit()

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds()

print(f"\n✅ Training completed in {training_time:.2f} seconds")
print(f"Final metrics: {result.metrics}")

## Evaluate Model


In [ ]:
# Get the trained model
model = result.checkpoint.get_model()

# Make predictions on test set
dtest = xgb.DMatrix(X_test)
y_pred_proba = model.predict(dtest)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate metrics
auc_score = roc_auc_score(y_test, y_pred_proba)
accuracy = (y_pred == y_test).mean()

print(f"Test AUC Score: {auc_score:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature importance
importance = model.get_score(importance_type='weight')
print("\nTop 10 Feature Importances:")
for feature, score in sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{feature}: {score}")

## Save Model to S3


In [ ]:
import tempfile
import json
from datetime import datetime

# Create model metadata
model_metadata = {
    'model_type': 'xgboost',
    'training_date': datetime.now().isoformat(),
    'feature_columns': feature_columns,
    'training_config': training_config,
    'metrics': {
        'auc_score': float(auc_score),
        'accuracy': float(accuracy)
    },
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

# Save model and metadata to S3
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_prefix = f"models/fraud_detection_xgboost_{timestamp}"

with tempfile.TemporaryDirectory() as temp_dir:
    # Save model
    model_path = f"{temp_dir}/model.xgb"
    model.save_model(model_path)
    
    # Save metadata
    metadata_path = f"{temp_dir}/metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(model_metadata, f, indent=2)
    
    # Upload to S3
    s3_client.upload_file(model_path, S3_BUCKET, f"{model_prefix}/model.xgb")
    s3_client.upload_file(metadata_path, S3_BUCKET, f"{model_prefix}/metadata.json")
    
    # Also save as "latest" for inference service
    s3_client.upload_file(model_path, S3_BUCKET, "models/latest/model.xgb")
    s3_client.upload_file(metadata_path, S3_BUCKET, "models/latest/metadata.json")

print(f"✅ Model saved to S3:")
print(f"- s3://{S3_BUCKET}/{model_prefix}/model.xgb")
print(f"- s3://{S3_BUCKET}/{model_prefix}/metadata.json")
print(f"- s3://{S3_BUCKET}/models/latest/model.xgb (for inference)")
print(f"- s3://{S3_BUCKET}/models/latest/metadata.json (for inference)")

## Test Inference


In [ ]:
# Test inference with a few samples
test_samples = X_test.head(5)
test_predictions = model.predict(xgb.DMatrix(test_samples))

print("Sample predictions:")
for i, (idx, row) in enumerate(test_samples.iterrows()):
    actual = y_test.loc[idx]
    predicted_prob = test_predictions[i]
    predicted_class = int(predicted_prob > 0.5)
    
    print(f"Sample {i+1}:")
    print(f"  Actual: {actual}, Predicted: {predicted_class}, Probability: {predicted_prob:.4f}")
    print(f"  Features: {dict(row)}")
    print()

## Cleanup


In [ ]:
# Shutdown Ray connection
ray.shutdown()
print("Ray connection closed")